# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the [FAIR^2 dataset](https://sen.science/doi/10.71728/senscience.qs2f-h81p) using the `mlcroissant` library. All data elements and references utilize their defined `@id`s, consistent with the Croissant schema for robust and reproducible analysis.

### Dataset Source
This dataset's Croissant schema is available at:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

The dataset contains tabular information on 77 cancer survivors with second primary colorectal cancer, including demographics, comorbidities, anatomical sites, treatments, intervals between diagnoses, histology, metastasis, and microsatellite instability (MSI-H) status.

In [ ]:
# Ensure mlcroissant is installed in the environment
!pip install mlcroissant --quiet

## 1. Data Loading

First, we load the dataset via its Croissant schema using `mlcroissant`. We also print out some key metadata properties.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Dataset Croissant schema URL (@id)
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset package
dataset = mlc.Dataset(croissant_url)

# Access the metadata (do not subscript!)
meta = dataset.metadata

print(f"Dataset Title: {meta.name}\n")
print(f"Description: {meta.description}\n")
print(f"Identifier: {meta.identifier}")
print(f"Version: {meta.version}")
print(f"License: {meta.license}")

## 2. Data Overview

Let's review the available record sets and fields included in this dataset. We list each record set and its fields by their `@id`, as per best practice.

In [ ]:
# List all record sets by @id with their fields
print('Available Record Sets:')
for rs in dataset.record_sets:
    print(f"- Record Set @id: {rs.id}")
    if hasattr(rs, 'name'):
        print(f"  Name: {rs.name}")
    if hasattr(rs, 'description'):
        print(f"  Description: {rs.description}")
    if hasattr(rs, 'fields') and rs.fields:
        print("  Fields:")
        for f in rs.fields:
            fname = getattr(f, 'name', '(unnamed)')
            print(f"   - @id: {f.id}\t name: {fname}")
    print('')

# Collect all record set @ids for later
record_set_ids = [rs.id for rs in dataset.record_sets]

if not record_set_ids:
    print('No record sets found in the dataset metadata.')

## 3. Data Extraction

We will extract data from each record set into pandas DataFrames, using only the Croissant `@id` to reference the record set. For illustration, we show column names for one main record set.

In [ ]:
# Extract records for each record set by @id
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded {len(df)} records from RecordSet {rs_id}")

# For demonstration, get columns of the first record set
if record_set_ids:
    first_rs = record_set_ids[0]
    print(f"\nColumns in RecordSet {first_rs}:")
    print(dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)

We demonstrate basic data processing steps, such as filtering on a numeric field (e.g., "Age at Second CRC") and normalizing this field. We use the correct field `@id` from the previous overview.

> _Update the `numeric_field_id` and `group_field_id` below based on the available columns printed above, using the relevant `@id`s from the metadata._

In [ ]:
# Select the record set to analyze (first one shown above)
record_set_id = record_set_ids[0] if record_set_ids else None

if record_set_id is not None:
    df = dataframes[record_set_id]
    
    # Let's inspect available columns (which correspond to field @id's)
    print('Columns:', df.columns.tolist())
    
    # Choose field to filter/normalize -- example using Age; Update '@id' if needed.
    # Let's find an 'age' field:
    age_field_id = None
    for c in df.columns:
        if 'age' in c.lower():
            age_field_id = c
            break
    if age_field_id is None:
        # fallback: choose first numeric column
        num_cols = df.select_dtypes(include=[np.number]).columns
        if len(num_cols) > 0:
            age_field_id = num_cols[0]
        else:
            print("No suitable numeric field found.")

    if age_field_id:
        print(f"Using numeric field: {age_field_id}")
        # Filter for age > 60 (clinical threshold)
        filtered = df[df[age_field_id] > 60]
        print(f"Filtered records where {age_field_id} > 60:")
        display(filtered.head())

        # Normalize the age field
        filtered[age_field_id + '_normalized'] = (
            (filtered[age_field_id] - filtered[age_field_id].mean()) / filtered[age_field_id].std()
        )
        print(f"\nNormalized {age_field_id}:")
        display(filtered[[age_field_id, age_field_id + '_normalized']].head())

        # Try grouping by a categorical field
        group_field_id = None
        for col in df.columns:
            if 'sex' in col.lower() or 'gender' in col.lower():
                group_field_id = col
                break
        if group_field_id:
            grouped = filtered.groupby(group_field_id)[age_field_id].mean()
            print(f"\nMean {age_field_id} by {group_field_id}:")
            display(grouped)
        else:
            print("\nNo grouping categorical field ('sex'/'gender') was found.")
    else:
        print("Field for numeric EDA not found in DataFrame.")
else:
    print("No record set available for analysis.")

## 5. Visualization

Let's visualize the distribution of the selected numeric field (e.g., age at second CRC diagnosis), and the relationship with a grouping categorical variable (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only proceed if EDA produced a filtered DataFrame and age_field_id
if 'filtered' in locals() and age_field_id is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered[age_field_id], bins=15, kde=True, color='cornflowerblue')
    plt.title(f'Distribution of {age_field_id}')
    plt.xlabel(age_field_id)
    plt.ylabel('Frequency')
    plt.show()
    if group_field_id is not None:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=filtered[group_field_id], y=filtered[age_field_id])
        plt.title(f'{age_field_id} by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(age_field_id)
        plt.show()
else:
    print("Visualization skipped: EDA did not find a suitable numeric field.")

## 6. Conclusion

In this notebook, we loaded and explored the FAIR^2 dataset of second primary colorectal cancers using the Croissant metadata standard and `mlcroissant` tooling. We demonstrated referencing all entities by their `@id`, basic EDA, and visualizations for numeric and categorical analysis. 

For further research, consult the field documentation using `meta` and reference the dataset schema for analysis pipelines or reproducible science.

_Remember to use the official field and record set `@id`s when accessing or sharing specific results to ensure consistency with FAIR and Croissant principles._